In [1]:
import requests

import geopandas as gpd

from sqlalchemy import create_engine

from dotenv import load_dotenv

import os

In [2]:
# Load environment variables containing database credentials and data source URLs
load_dotenv()

# Retrieve connection parameters from the .env file
DATABASE_URL = os.getenv('DATABASE_URL')
JPT_URL = os.getenv('JPT_URL')

# Create a SQLAlchemy engine for database operations
engine = create_engine(DATABASE_URL)

### Download Source Data


In [3]:
r_jpt = requests.get(JPT_URL, timeout=30)

with open('jpt.gml', 'wb') as f:
    f.write(r_jpt.content)

print(f'Status code: {r_jpt.status_code}')
print('JPT file downloaded successfully.')

Status code: 200
JPT file downloaded successfully.


### Load administrative boundaries from the GML file

In [4]:
df_jpt = gpd.read_file("jpt.gml")
print(f'Downloaded data for {len(df_jpt)} administrative units.')

Downloaded data for 410 administrative units.


### Prepare Warsaw District Boundaries

In [5]:
# Extract Warsaw district boundaries from the administrative units dataset
df_warsaw_districts = (
    df_jpt[df_jpt["JPT_NAZWA_"].str.endswith("dzielnica", na=False)]
    .copy()
    .reset_index(drop=True)
)

# Clean district names
df_warsaw_districts["district"] = (df_warsaw_districts["JPT_NAZWA_"].str.replace(" - dzielnica", ""))

# Keep only required columns
df_warsaw_districts = df_warsaw_districts[["district", "geometry"]]

print(f'Warsaw districts: {len(df_warsaw_districts)}.')

Warsaw districts: 18.


### Load District Boundaries to PostgreSQL

In [6]:
df_warsaw_districts.to_postgis(
    "dim_districts",
    engine,
    if_exists="replace",
    index=False
)

print(f'Loaded {len(df_warsaw_districts)} districts into PostgreSQL.')

Loaded 18 districts into PostgreSQL.
